# 04. Kurasi instruksi Indonesia dan persiapan SFT

Notebook ini memperlihatkan alasan kurasi, coverage, prompt, target PageSpec, provenance dan checklist review. Semua kandidat pilot masih **draft**. Run All tidak mengubah anotasi atau memulai training.

Baseline pretraining sudah selesai, tetapi probe JSON 0/4 valid. Kesiapan format draft tidak membuktikan kualitas target ataupun kemampuan model. Panduan lengkap: [review anotasi](../docs/ANNOTATION_REVIEW_GUIDE.md).

In [1]:
from pathlib import Path
import json
import csv
from collections import Counter
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
REPORT = ROOT / 'reports/data/ANNOTATION_REVIEW_PILOT_ID_V2.json'
report = json.loads(REPORT.read_text(encoding='utf-8'))
PACKET = ROOT / report['packet_path']
LOCAL_READY = PACKET.exists()
print('Interpreter:', sys.executable)
print('Paket lokal tersedia:', LOCAL_READY)
print('Status:', report['status'], '| reviewed produksi:', report['reviewed_production_rows'])
if not LOCAL_READY:
    print('Setelah clone, siapkan corpus dan draft lokal mengikuti panduan review. Ringkasan coverage tetap tersedia.')

Interpreter: F:\Generative_AI\Vibe_Coding_Proyek_1\.venv\Scripts\python.exe
Paket lokal tersedia: True
Status: awaiting_human_review | reviewed produksi: 0


## 1. Coverage awal dan pilot

Pilot dipilih dua contoh per kombinasi style dan partisi asal. Partisi diwarisi dari keluarga pretraining, bukan diacak ulang. Dua belas contoh hanya untuk memeriksa proses review; dataset harus diperluas untuk SFT dan evaluasi yang berarti.

In [2]:
display(pd.DataFrame([report['queue'], report['pilot']], index=['queue_awal', 'pilot']).T)
print('Context:', report['block_size'], 'token; panjang maksimum pilot:', report['pilot']['max_sequence_tokens'])
print('Interaksi tersedia:', report['pilot']['interaction_counts'])
print('Belum tersedia: testimonial, pricing_table, accordion, tabs, modal, filter.')

,queue_awal,pilot
rows,119,12
unique_families,102,12
split_counts,"{'train': 94, 'test': 14, 'validation': 11}","{'train': 4, 'validation': 4, 'test': 4}"
style_counts,"{'plain_css': 69, 'tailwind': 50}","{'plain_css': 6, 'tailwind': 6}"
source_counts,"{'websight_v01': 50, 'websight_v02': 50, 'vulc...","{'vulcan': 2, 'websight_v01': 4, 'websight_v02..."
section_counts,{'hero': 119},"{'contact_form': 1, 'navbar': 5, 'hero': 7, 'f..."
interaction_counts,{},"{'form_validate': 1, 'menu_toggle': 1}"
schema_valid_rows,119,12
within_context_rows,119,12
technical_ready_rows,0,12


Context: 1024 token; panjang maksimum pilot: 612
Interaksi tersedia: {'form_validate': 1, 'menu_toggle': 1}
Belum tersedia: testimonial, pricing_table, accordion, tabs, modal, filter.


## 2. Identitas dan checklist manusia

Kolom `human_decision`, `reviewer`, `reviewed_at` dan checklist sengaja kosong. `technical_ready_rows` hanya menyatakan prompt tidak kosong, berlabel Indonesia, schema benar, payload bisa di-encode, serta muat dalam context. Ketepatan bahasa dan makna tetap dinilai manusia.

In [3]:
drafts, worksheet = [], []
if LOCAL_READY:
    drafts = [json.loads(line) for line in (PACKET / 'drafts.jsonl').read_text(encoding='utf-8').splitlines()]
    with (PACKET / 'review.csv').open(encoding='utf-8-sig', newline='') as handle:
        worksheet = list(csv.DictReader(handle))
    columns = ['review_id', 'source_id', 'authoritative_split', 'style_mode', 'sequence_tokens', 'human_decision', 'reviewer']
    display(pd.DataFrame(worksheet)[columns])
    print('Keputusan manusia terisi:', sum(bool(row['human_decision'].strip()) for row in worksheet))
    print('Reviewer terisi:', sum(bool(row['reviewer'].strip()) for row in worksheet))
else:
    print('Worksheet dan draft lokal belum tersedia.')

,review_id,source_id,authoritative_split,style_mode,sequence_tokens,human_decision,reviewer
0,R-de355fb5b8b6,vulcan,train,plain_css,367,,
1,R-6a30eecac831,vulcan,train,plain_css,299,,
2,R-3fd49df7d7c4,websight_v01,validation,plain_css,610,,
3,R-df71eed31a9f,websight_v01,validation,plain_css,503,,
4,R-0f4b79fbd4da,websight_v01,test,plain_css,451,,
5,R-3e3a950af0ef,websight_v01,test,plain_css,429,,
6,R-488d3bfb598d,websight_v02,train,tailwind,588,,
7,R-c5a4ef0d5cf3,websight_v02,train,tailwind,564,,
8,R-f018074c02f9,websight_v02,validation,tailwind,336,,
9,R-b8e84893ce43,websight_v02,validation,tailwind,385,,


Keputusan manusia terisi: 0
Reviewer terisi: 0


## 3. Dua contoh dengan interaksi JavaScript

Sumber Vulcan mengandung validasi form dan menu responsif. Target hanya menyimpan ID interaksi yang diizinkan; source script tidak dimasukkan ke PageSpec. Untuk semua 12 perbandingan, buka paket v2 `packet.html`.

Navigasi Brand dibatasi sebagai navbar mandiri dengan satu tautan menuju section navbar. CTA EdTech menautkan fitur. Toko otomotif dan penerbitan menghapus tautan fungsi yang tidak tersedia. Pembatasan/proyeksi ini tetap perlu penilaian manusia; belum ada pemeriksaan browser.

In [4]:
if LOCAL_READY:
    for row in drafts:
        if row['target_spec']['interactions']:
            print('PROMPT:', row['prompt'])
            print(json.dumps(row['target_spec'], ensure_ascii=False, indent=2))
            print('CATATAN:', row['review_notes'])
            print('STATUS:', row['annotation_status'], '| reviewer:', repr(row['reviewer']))

PROMPT: Buat formulir kontak berbahasa Indonesia menggunakan CSS biasa. Gunakan tema hijau gelap dan font serif, kolom nama, email, serta pesan, tombol Kirim, dan validasi form dengan JavaScript.
{
  "schema_version": "1.0",
  "title": "Hubungi Kami",
  "language": "id",
  "style_mode": "plain_css",
  "theme": {
    "primary": "#10b981",
    "background": "#0c1f17",
    "text": "#ecfdf5",
    "font": "serif"
  },
  "sections": [
    {
      "id": "contact",
      "type": "contact_form",
      "props": {
        "heading": "Hubungi Kami",
        "fields": [
          {
            "name": "name",
            "label": "Nama",
            "type": "text"
          },
          {
            "name": "email",
            "label": "Email",
            "type": "email"
          },
          {
            "name": "msg",
            "label": "Pesan",
            "type": "textarea"
          }
        ],
        "submit_label": "Kirim"
      }
    }
  ],
  "interactions": [
    {
      "type": "

## Perbandingan revisi v1 dan v2

Empat target dan lima prompt diperbaiki. Identitas sumber, family assignment dan partisi dipertahankan. Pemeriksaan fragmen hanya memeriksa keberadaan ID section. Perubahan label/tujuan tautan dicatat manual dan tidak mendapat persetujuan manusia otomatis.

Paket v1 tetap tersedia sebagai riwayat. Gunakan worksheet v2 untuk review revisi terkini.

In [5]:
revisions = json.loads((ROOT / 'reports/data/ANNOTATION_PILOT_REVISIONS_V2.json').read_text(encoding='utf-8'))
display(pd.DataFrame(revisions['rows']))
print('Target berubah:', sum(row['target_changed'] for row in revisions['rows']))
print('Prompt berubah:', sum(row['prompt_changed'] for row in revisions['rows']))
print('Fragmen tidak tersedia sebelum/sesudah:', sum(row['unresolved_before'] for row in revisions['rows']), '/', sum(row['unresolved_after'] for row in revisions['rows']))
print('Identitas dan keluarga dipertahankan:', revisions['source_ids_and_families_preserved'])

,review_id,title,target_changed,prompt_changed,unresolved_before,unresolved_after
0,R-de355fb5b8b6,Hubungi Kami,False,False,0,0
1,R-6a30eecac831,Navigasi Brand,True,True,4,0
2,R-3fd49df7d7c4,Misi Organisasi Sosial,False,True,0,0
3,R-df71eed31a9f,Fitness Company,False,False,0,0
4,R-0f4b79fbd4da,Koleksi Fashion,False,False,0,0
5,R-3e3a950af0ef,Fitness Company,False,False,0,0
6,R-488d3bfb598d,Penyedia Layanan Kesehatan,False,False,0,0
7,R-c5a4ef0d5cf3,EdTech Startup,True,True,0,0
8,R-f018074c02f9,Aksesori Otomotif,False,False,0,0
9,R-b8e84893ce43,Katalog Mobil,False,False,0,0


Target berubah: 4
Prompt berubah: 5
Fragmen tidak tersedia sebelum/sesudah: 4 / 0
Identitas dan keluarga dipertahankan: True


## 4. Checksum, provenance dan partisi sumber

Pemeriksaan memakai family allocation aktual dan tokenizer proyek. Source HTML dibaca sebagai teks; tidak dijalankan. Tokenizer tidak dilatih ulang. File paket v2 adalah bukti awal yang tetap, sedangkan revisi worksheet sebaiknya disimpan ke paket baru.

In [6]:
from src.data.sources import sha256_file
from src.data.normalize import read_jsonl
from src.data.split import read_group_assignment, assignment_hash
from tokenizers import Tokenizer
from qa.prepare_annotation_review import inspect_candidate

checks = []
if LOCAL_READY and (ROOT / 'data/tokenizer/tokenizer.json').exists() and (ROOT / 'data/processed/pretrain/manifest.json').exists():
    for name, expected in report['files'].items():
        assert sha256_file(PACKET / name) == expected, f'Paket v2 berubah: {name}; gunakan revisi baru'
    assignment = read_group_assignment(ROOT / 'data/processed/pretrain')
    assert assignment_hash(assignment) == report['assignment_sha256']
    assert sha256_file(ROOT / 'data/tokenizer/tokenizer.json') == report['tokenizer_sha256']
    tokenizer = Tokenizer.from_file(str(ROOT / 'data/tokenizer/tokenizer.json'))
    wanted = {row['source_row_id'] for row in drafts}
    sources = {}
    for relative_path, expected in report['raw_files_sha256'].items():
        path = ROOT / relative_path
        assert sha256_file(path) == expected, f'Raw source berubah: {relative_path}'
        for row in read_jsonl(path):
            if row['source_row_id'] in wanted:
                sources[row['source_row_id']] = row
    for annotation in drafts:
        source = sources[annotation['source_row_id']]
        check = inspect_candidate(annotation, source, assignment, tokenizer, report['block_size'])
        assert not check['technical_blockers']
        checks.append(check)
    print('Verified candidates:', len(checks), '| source families:', len({row['source_group_id'] for row in checks}))
    print('Authoritative partitions:', dict(Counter(row['authoritative_split'] for row in checks)))
    first_source = sources[drafts[0]['source_row_id']]
    display({key: first_source[key] for key in ['source_row_id', 'source_revision', 'license_id', 'source_url', 'attribution']})
else:
    print('Paket/raw/tokenizer/split lokal dibutuhkan untuk memeriksa ulang provenance.')

Verified candidates: 12 | source families: 12
Authoritative partitions: {'train': 4, 'validation': 4, 'test': 4}


{'source_row_id': 'vulcan:19d2a482c78554e59feca0ce2b50fb7835e90c6f:13:4cfed126132b8fb7',
 'source_revision': '19d2a482c78554e59feca0ce2b50fb7835e90c6f',
 'license_id': 'mit',
 'source_url': 'https://huggingface.co/datasets/xlelords/vulcan',
 'attribution': 'Vulcan Front-End Code SFT Dataset, xlelords, 2026'}

## 5. Gate SFT dan target loss

Satu contoh memperlihatkan token prompt yang dimask dan target JSON yang mendapat loss. Memeriksa mask tidak menyetujui draft. Loader produksi menolak anotasi draft.

In [7]:
if checks:
    from src.data.dataset import encode_sft, PageSpecDataset
    first = drafts[0]
    target = json.dumps(first['target_spec'], ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    ids, labels = encode_sft(first['prompt'], target, tokenizer, report['block_size'])
    active = [index for index, label in enumerate(labels) if label != -100]
    print('Panjang input setelah padding:', len(ids))
    print('Label target aktif:', len(active), '| posisi target awal:', active[0])
    try:
        PageSpecDataset(drafts, tokenizer, 'sft', report['block_size'])
    except ValueError as error:
        print('Gate produksi menolak draft:', error)
    else:
        raise AssertionError('Draft tidak boleh diterima loader SFT produksi')

Panjang input setelah padding: 1024
Label target aktif: 287 | posisi target awal: 80
Gate produksi menolak draft: SFT dataset refuses draft/unreviewed records


## Kandidat perluasan coverage

Pemindaian membaca hanya partisi train dari corpus yang sudah disiapkan. Sinyal HTML/teks/JavaScript dipakai untuk menemukan bahan anotasi, termasuk kemungkinan placeholder. Kandidat belum memiliki prompt Indonesia, target PageSpec atau review. Angka nol tidak membuktikan tipe tersebut tidak ada di corpus; pencarian heuristik bisa melewatkannya.

In [8]:
coverage = json.loads((ROOT / 'reports/data/ANNOTATION_CANDIDATE_COVERAGE_V1.json').read_text(encoding='utf-8'))
display(pd.DataFrame({'signal_rows': coverage['candidate_row_counts'], 'signal_families': coverage['candidate_family_counts'], 'selected_for_inspection': coverage['selected_counts']}))
print('Baris train dibaca:', coverage['rows_scanned'], '| HTML:', coverage['html_rows_scanned'])
print('Kandidat tersimpan:', coverage['selected_records'], '| source/family unik:', coverage['selected_unique_source_rows'], '/', coverage['selected_unique_families'])
print('Status:', coverage['status'], '| tidak ada target SFT baru dari katalog ini.')

,signal_rows,signal_families,selected_for_inspection
testimonial,3379,3345,6
pricing_table,52,52,6
accordion,1,1,1
tabs,1,1,1
modal,0,0,0
filter,0,0,0


Baris train dibaca: 41639 | HTML: 38666
Kandidat tersimpan: 14 | source/family unik: 14 / 14
Status: candidate_signals_only | tidak ada target SFT baru dari katalog ini.


## 6. Langkah berikutnya

1. Bandingkan sumber, prompt, target, dan catatan perubahan pada paket.
2. Isi checklist, keputusan, nama reviewer, tanggal dan alasan. Revisi atau tolak kandidat yang tidak sesuai.
3. Simpan bukti review dan anotasi revisi baru. Hanya kandidat yang disetujui manusia diberi `annotation_status=reviewed` pada `reviewed.jsonl`.
4. Perluas coverage section/interaksi dan jumlah keluarga, tetap mengikuti partisi pretraining. Jangan gunakan test untuk tuning.
5. Persiapkan SFT setelah review, lalu gunakan parent `TRAIN-004-VERIFIED/best.pt`. Catat accepted/rejected, tokenizer, manifest, parent checkpoint dan TRAIN ID baru.

Tidak ada update optimizer atau SFT produksi yang dijalankan notebook ini. Aplikasi renderer/preview/download tetap tahap akhir.

In [9]:
print('Ringkasan tahap:', report['status'])
print('Reviewed produksi pada saat pembuatan paket:', report['reviewed_production_rows'])
print('Optimizer updates pada tahap kurasi ini: 0')
print('Panduan:', ROOT / 'docs/ANNOTATION_REVIEW_GUIDE.md')
print('Paket review:', PACKET / 'packet.html')

Ringkasan tahap: awaiting_human_review
Reviewed produksi pada saat pembuatan paket: 0
Optimizer updates pada tahap kurasi ini: 0
Panduan: F:\Generative_AI\Vibe_Coding_Proyek_1\docs\ANNOTATION_REVIEW_GUIDE.md
Paket review: F:\Generative_AI\Vibe_Coding_Proyek_1\outputs\annotation_review\pilot-id-v2\packet.html
